In [2]:
import chess

values = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0
}

def board_score(b):
    result = 0
    for p in values:
        result += len(b.pieces(p, chess.WHITE)) * values[p]
        result -= len(b.pieces(p, chess.BLACK)) * values[p]
    return result

def beam_strategy(b, width, level):

    active = [(b.copy(), [], board_score(b))]

    for _ in range(level):
        next_states = []

        for state, seq, val in active:

            for m in state.legal_moves:
                nb = state.copy()
                nb.push(m)

                sc = board_score(nb)

                next_states.append((nb, seq + [m], sc))

        next_states.sort(key=lambda x: x[2], reverse=True)

        active = next_states[:width]

    top = active[0]

    return top[1], top[2]

game = chess.Board()

path, value = beam_strategy(game, width=3, level=2)

print("Selected Move Path:", path)
print("Board Score:", value)

ModuleNotFoundError: No module named 'chess'

In [3]:
import random
import math

def calc_dist(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def path_length(path):
    total = 0
    for i in range(len(path)-1):
        total += calc_dist(path[i], path[i+1])
    total += calc_dist(path[-1], path[0])
    return total

def hill_search(nodes):

    route = nodes[:]
    random.shuffle(route)

    best_length = path_length(route)

    change = True

    while change:
        change = False

        for i in range(len(nodes)):
            for j in range(i+1, len(nodes)):

                temp_route = route[:]

                temp_route[i], temp_route[j] = temp_route[j], temp_route[i]

                new_len = path_length(temp_route)

                if new_len < best_length:
                    route = temp_route
                    best_length = new_len
                    change = True

    return route, best_length


cities = [
    (1,2),
    (4,5),
    (6,1),
    (3,7),
    (8,4),
    (2,6)
]

best_route, best_value = hill_search(cities)

print("Best Path Found:", best_route)
print("Path Cost:", best_value)

Best Path Found: [(1, 2), (2, 6), (3, 7), (4, 5), (8, 4), (6, 1)]
Path Cost: 20.60106358016498


In [4]:
import random
import math

locations = [
    (1,1),
    (3,4),
    (6,2),
    (2,7),
    (8,5),
    (4,1),
    (7,6),
    (5,8),
    (9,3),
    (2,5)
]

def calc_distance(x, y):
    return math.sqrt((x[0]-y[0])**2 + (x[1]-y[1])**2)

def route_cost(path):
    cost = 0
    for i in range(len(path)-1):
        cost += calc_distance(path[i], path[i+1])
    cost += calc_distance(path[-1], path[0])
    return cost

def score(path):
    return 1 / route_cost(path)

def choose(pop, scores):
    total = sum(scores)
    r = random.uniform(0, total)
    s = 0
    for p, sc in zip(pop, scores):
        s += sc
        if s > r:
            return p
    return pop[-1]

def combine(a, b):
    n = len(a)
    s, e = sorted(random.sample(range(n), 2))
    child = [None]*n
    child[s:e] = a[s:e]

    idx = 0
    for c in b:
        if c not in child:
            while child[idx] is not None:
                idx += 1
            child[idx] = c
    return child

def change(path, rate=0.1):
    p = path[:]
    if random.random() < rate:
        i, j = random.sample(range(len(p)), 2)
        p[i], p[j] = p[j], p[i]
    return p

def genetic_search(nodes, pop_size=40, gens=150):

    pop = [random.sample(nodes, len(nodes)) for _ in range(pop_size)]

    best = pop[0]
    best_cost = route_cost(best)

    for _ in range(gens):

        scores = [score(p) for p in pop]
        new_pop = []

        for _ in range(pop_size):
            p1 = choose(pop, scores)
            p2 = choose(pop, scores)
            child = combine(p1, p2)
            child = change(child)
            new_pop.append(child)

        pop = new_pop

        for p in pop:
            c = route_cost(p)
            if c < best_cost:
                best_cost = c
                best = p

    return best, best_cost

result_path, result_cost = genetic_search(locations)

print("Best Route Found:", result_path)
print("Route Length:", result_cost)

Best Route Found: [(7, 6), (2, 7), (5, 8), (2, 5), (3, 4), (1, 1), (4, 1), (6, 2), (9, 3), (8, 5)]
Route Length: 29.572329876258586


In [5]:
import copy

def beam_scheduler(jobs, units, width=3):
    start = {'plan':[[] for _ in range(units)], 'time':[0]*units}
    active = [start]

    for job in jobs:
        options = []
        for state in active:
            for u in range(units):
                new_state = copy.deepcopy(state)
                new_state['plan'][u].append(job['jid'])
                new_state['time'][u] += job['duration']

                weight = sum([j['importance'] for x in new_state['plan'][u]
                              for j in jobs if j['jid']==x])

                new_state['value'] = max(new_state['time']) - 0.1*weight
                options.append(new_state)

        options.sort(key=lambda x: x['value'])
        active = options[:width]

    best = active[0]
    return best['plan'], best['time']


jobs = [
    {'jid': 101, 'duration': 6, 'importance': 2},
    {'jid': 102, 'duration': 3, 'importance': 1},
    {'jid': 103, 'duration': 5, 'importance': 3},
    {'jid': 104, 'duration': 2, 'importance': 1},
    {'jid': 105, 'duration': 4, 'importance': 2},
    {'jid': 106, 'duration': 3, 'importance': 2}
]

units = 3

plan_result, time_result = beam_scheduler(jobs, units, width=3)

print("Final Job Distribution:", plan_result)
print("Execution Time per Unit:", time_result)

Final Job Distribution: [[101, 106], [102, 104], [103, 105]]
Execution Time per Unit: [9, 5, 9]
